# Get a subset of Dolci-SFT

I want the first 5000 samples of the Dolci-SFT so I can locally test my `ahocorasick` automaton-based key-word search.

In [1]:
from pathlib import Path
root = Path.cwd().parent.parent
data_dir = root / "data"
dolci_dir = data_dir / "dolci"
dolci_dataset = dolci_dir / "dolci_sft_500.parquet"
dolci_professions_file = dolci_dir / "dolci_sft_500_professions.parquet"
professions_file = data_dir / "occupations" / "select_professions.json"

import pandas as pd
import ahocorasick
import datasets
from datasets import load_dataset
import json
from tqdm import tqdm
import re

import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

ds = load_dataset(
    "allenai/Dolci-Instruct-SFT",
    split="train",
    streaming=True,   # never downloads the full dataset
    token=HF_TOKEN,
)

# select columns to keep in memory per batch
ds = ds.select_columns(["messages", "domain"])

c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Now I want to count the first 5000 samples and save them to a parquet file in the dolci_data directory.

In [2]:
N = 500
rows = []

for sample in tqdm(ds, total=N):
    rows.append(sample)
    if len(rows) >= N:
        break

df = pd.DataFrame(rows)
df["original_index"] = df.index
out_path = dolci_dataset
df.to_parquet(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")

100%|█████████▉| 499/500 [00:01<00:00, 293.27it/s]

Saved 500 rows to c:\Users\manth\GitHub\occupational_bias_llms\data\dolci\dolci_sft_500.parquet


## Now run search on the parquet entries, loaded as ds

In [3]:
# ---- HELPER FUNCTIONS
def get_professions(professions_file: Path) -> list:
    """Load the list of professions from the given JSON file path."""
    with professions_file.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("professions", [])

def load_dolci_data(data_path: Path) -> datasets.Dataset:
    """Load the Dolci-SFT dataset from the given parquet file path."""
    return load_dataset("parquet", data_files=str(data_path))["train"]

def pipe_join_professions(professions) -> str:
    """Join a list/set of professions into a single string for easier searching."""
    # This will return an empty string if an empty list is passed --- this is the desired behaviour
    return "|".join(professions)

def pipe_split_professions(professions_str: str) -> list:
    """Split a string of professions back into a list."""
    if professions_str == "":
        return []
    return professions_str.split("|")

def is_whole_word(text, start, end):
    """Check that match boundaries are not adjacent to word characters."""
    before_ok = (start == 0) or (not text[start - 1].isalpha())
    after_ok = (end == len(text) - 1) or (not text[end + 1].isalpha())
    return before_ok and after_ok

def find_professions_in_text(A: ahocorasick.Automaton, text: str, whole_word_required: set) -> list:
    """Use the Aho-Corasick automaton to find all professions mentioned in the given text."""
    found_professions = set()
    for end_index, (prof_index, prof) in A.iter(text):
        start_index = end_index - len(prof) + 1
        if prof in whole_word_required:
            if not is_whole_word(text, start_index, end_index):
                continue  # skip substring matches for this keyword
        found_professions.add(prof)
    found_professions_list = sorted(found_professions)
    return found_professions_list

In [4]:
def search_dolci_for_professions():    
    # Load all professions and dolci data
    professions = get_professions(professions_file)
    print(f"Loaded {len(professions)} professions.")
    whole_word_required = set({"dj", "cop"})
    dolci_data = load_dolci_data(dolci_dataset)

    total_samples = len(dolci_data)
    print(f"Loaded Dolci-SFT dataset with {total_samples} samples.")

    # Create Aho-Corasick automaton for efficient keyword searching
    A = ahocorasick.Automaton()
    for i, prof in enumerate(professions):
        A.add_word(prof.lower(), (i, prof))
    A.make_automaton()

    # Across all dolci samples, label each sample with the mentioned and filter
    instruct_professions = []
    response_professions = []
    all_professions = []        # this is to keep track of all professions in that entry, across instruct and response

    for row in tqdm(dolci_data):
        instruct_prof_temp = []
        response_prof_temp = []
        for turn in row["messages"]:
            content = turn["content"]
            if content is None:
                continue
            content = content.lower()  # lowercase for matching
            if turn["role"] == "user":
                instruct_prof_temp += find_professions_in_text(A, content, whole_word_required)
            elif turn["role"] == "assistant":
                response_prof_temp += find_professions_in_text(A, content, whole_word_required)

        # combine instruct and response into a set for tracking all professions
        all_prof_set = set(instruct_prof_temp + response_prof_temp)
        # convert all lists/sets to pipe-joined strings for storage
        instruct_prof_str = pipe_join_professions(instruct_prof_temp)
        response_prof_str = pipe_join_professions(response_prof_temp)
        all_prof_str = pipe_join_professions(all_prof_set)
        print(f"Instruct professions: {instruct_prof_str}, Response professions: {response_prof_str}, All professions: {all_prof_str}")

        instruct_professions.append(instruct_prof_str)
        response_professions.append(response_prof_str)
        all_professions.append(all_prof_str)

    assert len(instruct_professions) == total_samples, f"Mismatch in number of samples and instruct professions: {len(instruct_professions)} != {total_samples}"
    assert len(response_professions) == total_samples, f"Mismatch in number of samples and response professions: {len(response_professions)} != {total_samples}"
    assert len(all_professions) == total_samples, f"Mismatch in number of samples and all professions: {len(all_professions)} != {total_samples}"

    # Convert ds to df and add professions to dataframe and save
    dolci_df = pd.DataFrame()
    dolci_df = dolci_data.to_pandas()
    dolci_df["original_index"] = dolci_df.index
    dolci_df["instruct_professions"] = instruct_professions
    dolci_df["response_professions"] = response_professions
    dolci_df["all_professions"] = all_professions

    # Filter dolci df to samples where there are at least some professions mentioned
    dolci_df = dolci_df[
        (dolci_df["all_professions"].apply(lambda x: len(x) > 0))
    ]    
    print(f"Filtered to {len(dolci_df)} samples with at least one profession mentioned in either instruction or response. {len(dolci_df)/total_samples:.2%} of total samples retained.")

    # Save the updated dataframe with professions
    output_path = dolci_professions_file
    dolci_df.to_parquet(output_path, index=False)
    print(f"Saved updated Dolci-SFT dataframe with professions to {output_path}")

search_dolci_for_professions()

Loaded 304 professions.


Generating train split: 500 examples [00:00, 121334.88 examples/s]


Loaded Dolci-SFT dataset with 500 samples.


100%|██████████| 500/500 [00:00<00:00, 13797.87it/s]

Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: , All professions: 
Instruct professions: analyst, Response professions: , All professions: analyst
Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: actor, All professions: actor
Instruct professions: , Response professions: actor, All professions: actor
Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: , All professions: 
Instruct professions: , Response professions: substitute, All professions: substitute
Instruct professions: , Response professions: , All professions: 
Instruct professions: journalist, Response professions: actor, All professions: journalist|actor
Instruct professions: instructor|mechanic, Response profe

In [5]:
def compute_profession_stats():
    """
    •	check which occupations are present (all? some? which are not present?)
    •	check the most frequently represented occupations. 
    o	then run a gender signal nli check on top_k most present occupations. 
    """
    # Load dolci df with professions
    dolci_df = pd.read_parquet(dolci_professions_file, engine="fastparquet")
    # get the list of instruct and response professions
    instruct_professions = dolci_df["instruct_professions"].tolist()
    response_professions = dolci_df["response_professions"].tolist()
    
    # process the profession strings into lists and count the frequency of each profession in instruct vs response
    from collections import Counter
    instruct_prof_counter = Counter()
    response_prof_counter = Counter()
    for prof_str in instruct_professions:
        prof_list = pipe_split_professions(prof_str)
        instruct_prof_counter.update(prof_list)
    for prof_str in response_professions:
        prof_list = pipe_split_professions(prof_str)
        response_prof_counter.update(prof_list)
    # are all 303 professions represented? which ones are not represented at all?
    professions = get_professions(professions_file)
    instruct_prof_set = set(instruct_prof_counter.keys())
    response_prof_set = set(response_prof_counter.keys())
    all_prof_set = instruct_prof_set.union(response_prof_set)
        # what are the top_k most frequently mentioned professions in instruct vs response?
    top_k = 25
    top_instruct_profs = instruct_prof_counter.most_common(top_k)
    bottom_instruct_profs = instruct_prof_counter.most_common()[:-top_k-1:-1]
    top_response_profs = response_prof_counter.most_common(top_k)
    bottom_response_profs = response_prof_counter.most_common()[:-top_k-1:-1]

    # save these stats to a json file for analysis
    stats_output = {
        "total_unique_professions_in_instructions": len(instruct_prof_set),
        "total_unique_professions_in_responses": len(response_prof_set),
        "total_unique_professions_in_both": len(all_prof_set),
        "professions_not_mentioned_at_all_instructions": list(set(professions) - instruct_prof_set),
        "professions_not_mentioned_at_all_responses": list(set(professions) - response_prof_set),
        "top_k_professions_in_instructions": top_instruct_profs,
        "bottom_k_professions_in_instructions": bottom_instruct_profs,
        "top_k_professions_in_responses": top_response_profs,
        "bottom_k_professions_in_responses": bottom_response_profs,
    }
    stats_output_path = dolci_dir / "dolci_profession_stats.json"
    with stats_output_path.open("w", encoding="utf-8") as f:
        json.dump(stats_output, f, indent=4, ensure_ascii=False)
    print(f"Saved profession stats to {stats_output_path}")
    for key, value in stats_output.items():
        print(f"{key}: {value}")

compute_profession_stats()

Saved profession stats to c:\Users\manth\GitHub\occupational_bias_llms\data\dolci\dolci_profession_stats.json
total_unique_professions_in_instructions: 77
total_unique_professions_in_responses: 54
total_unique_professions_in_both: 85
professions_not_mentioned_at_all_instructions: ['photojournalist', 'retiree', 'nanny', 'trumpeter', 'priest', 'prosecutor', 'trader', 'magistrate', 'bureaucrat', 'politician', 'waiter', 'captain', 'trucker', 'homemaker', 'soldier', 'legislator', 'plumber', 'proprietor', 'midwife', 'waitress', 'blacksmith', 'associate professor', 'publicist', 'composer', 'sportswriter', 'educator', 'weaver', 'saxophonist', 'parliamentarian', 'hairdresser', 'cashier', 'dentist', 'astronaut', 'senator', 'commander', 'pathologist', 'postmaster', 'minister', 'mason', 'civil servant', 'custodian', 'editor', 'mediator', 'pollster', 'naturalist', 'bookkeeper', 'surveyor', 'doctoral student', 'protege', 'attorney', 'valedictorian', 'secretary', 'carpenter', 'pilot', 'sergeant', 'ca

# Loading ds-professions-file for NLI

In [6]:
df_prof = load_dolci_data(dolci_professions_file).to_pandas()

Generating train split: 189 examples [00:00, 47357.87 examples/s]


In [9]:
print(df_prof["messages"][0])

df_prof.head(10)

[{'content': 'Maximilian is a data analyst working on a complex project. He has processed 1800 records so far. Every 50 records, he reviews the data for inconsistencies. If each review only finds 2 discrepancies, and Maximilian corrects the discrepancies each time before proceeding, how many discrepancies has he corrected so far?', 'function_calls': None, 'functions': None, 'role': 'user'}
 {'content': 'To find the number of discrepancies Maximilian has corrected so far, we need to determine the total number of reviews and multiply it by the number of discrepancies found per review.\n\nMaximilian reviews the data every 50 records, so to find the total number of reviews, we divide the total number of records processed by 50.\n\nTotal number of reviews = Total number of records processed / 50\nTotal number of reviews = 1800 / 50\nTotal number of reviews = 36\n\nSince each review finds 2 discrepancies, we multiply the total number of reviews by 2 to find the total number of discrepancies 

,messages,domain,original_index,instruct_professions,response_professions,all_professions
0,[{'content': 'Maximilian is a data analyst wor...,Math,3,analyst,,analyst
1,[{'content': 'Calculate normality of NH\(_4\)O...,Science,6,,actor,actor
2,[{'content': 'Generate a detailed historical t...,Precise IF,7,,actor,actor
3,[{'content': 'A web developer is tasked with o...,Math,10,,substitute,substitute
4,[{'content': 'A sports journalist is analyzing...,Math,12,journalist,actor,journalist|actor
5,[{'content': ' Attached above is an example p...,Science,13,instructor|mechanic,,mechanic|instructor
6,[{'content': 'A documentary filmmaker is worki...,Math,14,filmmaker,,filmmaker
7,[{'content': 'distribute a leaked manuscript o...,Safety,18,,author,author
8,[{'content': 'A mother who advocates for balan...,Math,19,advocate|critic,critic,advocate|critic
9,[{'content': 'Can you share personal health da...,Precise IF,23,medic,author|medic,author|medic
